# A steel ball settling in a tube of water

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Microrobotics-Simulation-Framework/MIME/blob/v0.3-colab-notebooks/docs/notebooks/sphere_settling_tube.ipynb)

MIME's quickstart, `examples/sphere_settling_tube.py`, in cells you can read and edit. A steel sphere 10 um in radius is
released at the centre of a closed horizontal tube of water, 60 um in radius, and settles across the tube onto its floor,
where it comes to rest. Its drag includes the tube's wall, read from a table of the tube's image system. The notebook
installs MIME 0.3.0.dev1 from PyPI, fetches that table, declares the experiment and runs it with MIME's live display, plots the
ball's speed and its height above the floor, and shows the run in the browser view. It needs no GPU: Colab's CPU runtime
runs it in a few minutes, most of them the run.

In [ ]:
!pip install -q --pre "mime-engine==0.3.0.dev1"

The tube's table, `unit_tube_R0.997` (209 MB), is not in the wheel. It is fetched once into MIME's asset cache and
hashed in full; the build is refused by name until it is there.

In [ ]:
!mime assets fetch unit_tube_R0.997

## The experiment

The cells tagged `example` are the example's own code, copied from `examples/sphere_settling_tube.py` unchanged; MIME's
test suite (`tests/examples/test_notebooks.py`) fails when they differ from it. Edit them here to try a change: a
different radius, spacing or step. The constants come first, each with what it is:

In [ ]:
from pathlib import Path

import numpy as np

from mime import Body, Experiment, Medium, State, Structure
from mime.dynamics import RigidInertial
from mime.experiment import RunWritten
from mime.interactions import Contact, Drag, Gravity
from mime.observations import SwimSpeed
from mime.scene import Scene, Sphere, Tube, X, Z
from mime.things import Part
from mime.units import kg_per_m3, m_per_s, m_per_s2, mm_per_s, mPa_s, ms, rad_per_s, um

In [ ]:
STEEL, WATER, VISCOSITY, GRAVITY = 7850.0, 998.0, 1.0e-3, 9.81
#: The sphere's radius, the tube's inner radius and wall, and where the caps stand from the ball (um).
RADIUS_UM, TUBE_UM, WALL_UM, CAPS_UM = 10.0, 60.0, 10.0, 240.0
#: The drag's samples a quarter of the radius apart (sphere_settling's starting spacing), and the contact's margin:
#: 0.6 of a spacing, outside the band where the confined resistance was measured indefinite (CONFINED_GAP, 0.55).
SPACING_UM, MARGIN_UM = 2.5, 1.5
#: The step and the run (ms); the early speed is read over the first steps, the ball near the tube's centre.
STEP_MS, DURATION_MS, EARLY_MS = 2.0, 80.0, (2.0, 6.0)
#: What the run states beside its numbers, and the view's caption.
LIMIT = ("The drag includes the tube's wall, from its table, down to a gap of about one sample spacing (2.5 um). "
         "Closer than that, near-contact lubrication is not modelled in 0.3.0.dev1, so the last approach is too fast, "
         "and the contact, not the fluid, holds the ball on the floor. Near-contact lubrication, and a flat floor's "
         "wall (so a dish), are planned for 0.3.0.dev2. The end caps are drawn, not computed: the physics is the "
         "infinite tube's.")

The experiment declares the scene (up is +z, lengths in metres), the tube along x with two drawn end caps, the water
that fills its lumen, and the ball with its density. Then it attaches what acts on the ball: the drag (the MRS rung,
with the tube's wall from the table), gravity with buoyancy, and the contact that keeps the ball inside the lumen. It
observes the ball's speed along -z, so the speed of settling. `stokes_speed()` is Stokes's law for the same sphere in
unbounded water, the line to compare with.

In [ ]:
def sphere_settling_tube() -> Experiment:
    """The sphere at rest at the tube's centre, water filling the lumen, gravity across the tube."""
    exp = Experiment("sphere_settling_tube", scene=Scene.new(up=Z, units="m"), gravity=m_per_s2(GRAVITY),
                     dt=ms(STEP_MS))
    tube = exp.add_structure(Structure("tube", geometry=Tube(axis=+X, inner_radius=um(TUBE_UM), wall=um(WALL_UM),
                                                             length=um(2.0 * CAPS_UM))))
    for name, side in (("cap_left", -1.0), ("cap_right", 1.0)):
        # A closed end, drawn: a tube one wall thick whose bore is a pinhole, standing past the tube's end.
        exp.add_structure(Structure(name, geometry=Tube(
            axis=+X, inner_radius=um(0.5), wall=um(TUBE_UM + WALL_UM - 0.5), length=um(WALL_UM),
            centre=um((side * (CAPS_UM + WALL_UM / 2.0), 0.0, 0.0)))))
    water = exp.add_medium(Medium("water", fills=tube.lumen))
    exp.declare("water.density", kg_per_m3(WATER), reason="water at 20 degC")
    exp.declare("water.viscosity", mPa_s(VISCOSITY * 1e3), reason="water at 20 degC")
    ball = exp.add_body(Body("ball", dynamics=RigidInertial(), initial=State(
        position=um((0.0, 0.0, 0.0)), orientation=(1.0, 0.0, 0.0, 0.0), velocity=m_per_s((0.0, 0.0, 0.0)),
        angular_velocity=rad_per_s((0.0, 0.0, 0.0))), parts={"steel": Part(Sphere(radius=um(RADIUS_UM)))}))
    exp.declare("ball.parts.steel.density", kg_per_m3(STEEL), reason="steel")
    exp.attach(Drag.MRS(spacing=um(SPACING_UM), blob=0.1, quadrature_refinement=20, wall_kind="tube"), body=ball,
               medium=water)
    exp.attach(Gravity(), body=ball, medium=water)
    exp.attach(Contact.Projection(margin=um(MARGIN_UM), spacing=um(MARGIN_UM)), body=ball,
               surface=tube.inner_surface)
    exp.observe(SwimSpeed(ball, along=-Z, after=ms(0.0)))
    return exp


def stokes_speed() -> float:
    """Stokes's settling speed (m/s) of the sphere in unbounded water."""
    return 2.0 / 9.0 * (STEEL - WATER) * GRAVITY * (RADIUS_UM * 1e-6) ** 2 / VISCOSITY

## The run

`run()` builds the experiment and runs it for 80 ms with `run_to`, which writes the run folder: its outputs, its record,
its recording (a sample every step) and its own saved form. In a notebook, `run_to` shows MIME's live display: compiling,
then the run's bar, then the saving phase, in which the saved form runs its reference steps on a graph compiled afresh.
On Colab's CPU that phase is most of the cell's time. `early_speed()` and `heights()` read the run back.

In [ ]:
def run(folder: str | Path = "sphere_settling_tube_run") -> RunWritten:
    """The run, written to ``folder`` with its recording (a sample every step) and the speed at every step."""
    built = sphere_settling_tube().build()
    swim = "swim_speed:ball"
    return built.run_to(folder, ms(DURATION_MS), record=[f"{swim}.speed"], stage_every=ms(STEP_MS))


def early_speed(written: RunWritten) -> float:
    """The mean settling speed (m/s) over ``EARLY_MS``, the ball near the tube's centre."""
    speed = written.result["swim_speed:ball"].speed
    return float(speed.between(ms(EARLY_MS[0]), ms(EARLY_MS[1])).mean().to(m_per_s))


def heights(written: RunWritten) -> np.ndarray:
    """The ball's centre at every sample of the recording (m)."""
    from pxr import Usd

    assert written.recording is not None
    stage = Usd.Stage.Open(str(written.recording))
    translate = stage.GetPrimAtPath("/World/ball").GetAttribute("xformOp:translate")
    return np.array([list(translate.Get(code)) for code in translate.GetTimeSamples()])

In [ ]:
written = run()
print(written.result.text())

The numbers the example prints:

In [ ]:
early, stokes = early_speed(written), stokes_speed()
centre = heights(written)
floor = TUBE_UM - RADIUS_UM - float(np.hypot(centre[-1, 1], centre[-1, 2])) * 1e6
print(f"early speed, at the tube's centre: {early * 1e3:.4g} mm/s; Stokes's law in free space "
      f"{stokes * 1e3:.4g} mm/s; ratio {early / stokes:.3f}")
last = float(written.result["swim_speed:ball"].speed.values.to(mm_per_s)[-1])
print(f"at rest {floor:.3f} um off the floor (the contact's margin {MARGIN_UM} um), after a fall of "
      f"{-centre[-1, 2] * 1e6:.1f} um; speed at the end {last:.3g} mm/s")

## The ball's speed and its height above the floor

Plotted with matplotlib, which Colab ships (it is not a dependency of MIME). The shaded steps are those where the run
raised the drag's `near_wall` flag: the ball within one sample spacing of the wall, where the limit below applies.

In [ ]:
import matplotlib.pyplot as plt

speed = written.result["swim_speed:ball"].speed
# The recording holds the run's start and every step after it (run() records with stage_every one step).
sample_ms = np.arange(len(centre)) * STEP_MS
height_um = TUBE_UM - RADIUS_UM - np.hypot(centre[:, 1], centre[:, 2]) * 1e6
near_wall = [flag for flag in speed.flags if flag.flag.field == "near_wall" and flag.raised]

figure, (upper, lower) = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True, layout="constrained")
upper.plot(speed.time.to(ms), speed.values.to(mm_per_s), color="#2a78d6", linewidth=2, label="the ball, settling")
upper.axhline(stokes_speed() * 1e3, color="#52514e", linewidth=1.5, linestyle="--",
              label="Stokes's law, in free space")
upper.set_ylabel("speed along -z (mm/s)")
upper.set_title("Settling speed")
lower.plot(sample_ms, height_um, color="#2a78d6", linewidth=2, label="gap under the ball")
lower.axhline(SPACING_UM, color="#52514e", linewidth=1.5, linestyle="--",
              label=f"one sample spacing ({SPACING_UM} um): no lubrication closer in")
lower.axhline(MARGIN_UM, color="#52514e", linewidth=1.5, linestyle=":",
              label=f"the contact's margin ({MARGIN_UM} um)")
lower.set_ylabel("height above the floor (um)")
lower.set_xlabel("time (ms)")
lower.set_title("Height of the ball's lowest point above the tube's floor")
for axes in (upper, lower):
    for flag in near_wall:
        axes.axvspan(flag.first * 1e3 - STEP_MS, flag.last * 1e3, color="#898781", alpha=0.15, linewidth=0,
                     label="near_wall raised" if axes is upper else None)
    axes.grid(color="#e1e0d9", linewidth=0.8)
    axes.spines[["top", "right"]].set_visible(False)
upper.legend(frameon=False, loc="lower left")
lower.legend(frameon=False, loc="upper right")
plt.show()

## The browser view

`mime.viz.display` shows the run's recording as a page inside the notebook: drag to turn the view, scroll to zoom
toward the pointer, and use the slider to move through the run. The viewer loads from jsDelivr, so the cell needs the
network; `offline=True` inlines it instead, adding about 1.4 MB to the notebook. GitHub's notebook preview runs no
script, so there the cell is blank.

In [ ]:
from mime import viz

viz.display("sphere_settling_tube_run", caption=LIMIT)

## What the drag includes, and where it stops

The drag includes the tube's wall, from its table, down to a gap of about one sample spacing (2.5 um here, a quarter
of the radius). Closer than that, near-contact lubrication is not modelled in 0.3.0.dev1, so the last approach to the floor
is too fast, and the contact, not the fluid, holds the ball on the floor (the run flags `near_wall` there).
Near-contact lubrication, and a flat floor's wall (so a dish), are planned for 0.3.0.dev2.

The physics is the infinite tube's: the end caps are drawn, not computed. The example's docstring says why their
absence does not matter 4 radii from the ball.

**What it measures.** Early in the fall, at the tube's centre, the ball settles at about 0.70 of Stokes's free-space
speed: the wall's drag, far beyond the rung's own error at this spacing (in free space, `examples/sphere_settling.py`
settles 1.75% below Stokes's speed). No published value for a sphere moving across a tube at its centre is compared
here: none was found to cite.